# 🔍 6. After-Agent Guardrail

An **after-agent guardrail** runs after the agent has completed its response but before the output reaches the user.

It can inspect the generated response and block or modify unsafe content.

---

## 🔄 How After-Agent Guardrails Work

```text
User Input
    ↓
🤖 Agent
    ↓
Generated Output
    ↓
After-Agent Guardrail
    ↓
Output Safe?
    ├── Yes → ✅ Return to User
    └── No  → 🚫 Block / Modify

## 🔍 After-Agent Guardrail Example

A model-based guardrail can be used to evaluate the agent's output.

Unlike the before-agent guardrail, this approach uses an LLM to understand the generated response and identify potentially unsafe content.

### Import Required Libraries

In [1]:
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain_groq import ChatGroq
from langgraph.runtime import Runtime

### Initialize Groq Models

In [2]:
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

safety_model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

### Create Safety Guardrail Middleware

In [4]:
class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Evaluate the agent's output
    before it reaches the user.
    """

    def __init__(self, safety_model):
        super().__init__()
        self.safety_model = safety_model

    @hook_config()
    def after_agent(
        self,
        state: AgentState,
        runtime: Runtime
    ) -> dict[str, Any] | None:

        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        output = last_message.content

        prompt = f"""
Evaluate the following AI-generated response for safety.

Reply with only:
SAFE
or
UNSAFE

Response:
{output}
"""

        result = self.safety_model.invoke([
            {
                "role": "user",
                "content": prompt
            }
        ])

        verdict = result.content.strip().upper()

        if "UNSAFE" in verdict:
            print("🚫 Unsafe output detected.")

            return {
                "messages": [{
                    "role": "assistant",
                    "content": (
                        "I’m sorry, but I cannot provide that response."
                    )
                }]
            }

        print("✅ Output passed safety check.")

        return None

### Create Agent with After-Agent Guardrail

In [5]:
safety_guardrail = SafetyGuardrailMiddleware(
    safety_model=safety_model
)

safe_agent = create_agent(
    model=model,
    tools=[],
    middleware=[safety_guardrail],
)

print("Agent with after-agent safety guardrail created!")

Agent with after-agent safety guardrail created!


### Test the Guardrail

In [6]:
result = safe_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Explain how to stay safe online."
        }
    ]
})

print("=== Final Response ===")
print(result["messages"][-1].content)

✅ Output passed safety check.
=== Final Response ===
**Staying safe online is a combination of good habits, solid tools, and a bit of vigilance. Below is a practical, step‑by‑step guide you can follow to protect your personal information, devices, and digital reputation.**

---

## 1. Secure Your Accounts

| Action | Why It Matters | How to Do It |
|--------|----------------|--------------|
| **Use strong, unique passwords** | Prevents attackers from guessing or reusing credentials across sites. | • 12‑+ characters<br>• Mix of upper/lowercase, numbers, symbols<br>• No personal info (birthdays, names)<br>• Use a **password manager** (e.g., Bitwarden, 1Password, LastPass) to generate and store them. |
| **Enable Two‑Factor Authentication (2FA)** | Adds a second verification step, so a stolen password alone isn’t enough. | • Prefer **authenticator apps** (Google Authenticator, Authy, Microsoft Authenticator) over SMS.<br>• For high‑value accounts, consider **hardware keys** (YubiKey, Goog

## 💡 Key Idea

> **After-agent guardrails inspect the agent's generated output before it reaches the user.**

This provides an additional safety layer even when the input itself was allowed.

---

## 📝 Key Takeaways

- An **after-agent guardrail** runs after the agent generates a response.
- It can inspect the output before it reaches the user.
- Model-based guardrails can evaluate output using an LLM.
- They are useful for **output safety, compliance, and quality validation**.
- Unsafe responses can be blocked or replaced with a safe message.
- Combining before-agent and after-agent guardrails provides **layered protection**.